In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
# We pull a version that has already been compressed to 4-bit by the community
model_id = "Qwen/Qwen2.5-14B-Instruct-AWQ"

print("Downloading pre-quantized blocks directly to GPU...")

# 1. Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# 2. Stream the 4-bit files straight into your hardware
# This will bypass your 12GB system RAM limitation and fit easily into your 16GB VRAM
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,
    device_map="cuda" # Automatically targets your GPU
)

print("Model successfully loaded onto your GPU! Ready to chat.")

In [ ]:
# 3. Use the model immediately
messages = [
    {"role": "system", "content": "Translate the following English social-media text into natural Indonesian. Preserve the original meaning, tone, ambiguity, slang, exaggeration, sarcasm, and pragmatic cues as closely as possible. Do not explain the text, resolve ambiguity, infer unstated meaning, or add information. Return only the translated text."},
    {"role": "user", "content": "What a successful toast, it looks so delicious!"}
]

# Format prompt using Qwen's specific template structure
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# Generate response
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512
)

# Extract only the newly generated text fragments
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
print("\n--- Model Response ---")
print(response[0])


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
model_name = "Qwen/Qwen2.5-14B-Instruct-AWQ"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]
response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
